# 一个样本的 DOME + Flow Matching 训练流水线

这个 notebook 按 `tools/train_diffusion.py` 里的真实训练顺序，解释一个 batch 中的一个样本如何进入模型：

1. 从 dataloader 取出 `input_occs, target_occs, metas`。
2. VAE encoder 把 occupancy 序列压成 latent。
3. 整理 latent 维度为 DOME 需要的 `(B, F, C, H, W)`。
4. 根据概率决定是否使用 pose 条件。
5. Flow Matching 采样 `(t, x_t, u_t)`。
6. DOME 输入 `(x_t, t, metas)`，预测速度场 `v_t`。
7. 用 `MSE(v_t, u_t)` 训练。
8. 反向传播、梯度裁剪、optimizer step、EMA 更新。

前半部分使用 dummy 张量和 dummy 模型，可以在没有 nuScenes 数据和 checkpoint 的情况下理解 shape 与目标。后半部分给出真实模型单 batch 调试模板。

## 0. 环境和路径

建议用项目环境启动：

```bash
conda activate torchcfm
cd /mnt/data2/whz/dome-cfm  # 服务器示例
jupyter notebook train_pipeline_one_sample.ipynb
```

如果在本机打开，下面的 `PROJECT_ROOT` 会自动使用 notebook 所在目录；如果路径不对，手动改成项目根目录。

In [1]:
# 基础导入与项目路径设置
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'tools' / 'train_diffusion.py').exists():
    # 如果 notebook 不是从项目根目录启动，可以在这里手动指定。
    PROJECT_ROOT = Path('/mnt/data2/whz/dome-cfm')

sys.path.insert(0, str(PROJECT_ROOT))
print('项目根目录:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])
print('解释器:', sys.executable)

for name in ['torch', 'torchcfm', 'einops', 'mmengine', 'mmcv', 'timm']:
    print(f'{name}:', 'OK' if importlib.util.find_spec(name) else 'MISSING')

项目根目录: /mnt/data2/whz/dome-cfm
Python: 3.8.0
解释器: /mnt/data2/congfeng/miniconda3/envs/torchcfm/bin/python
torch: OK
torchcfm: OK
einops: OK
mmengine: OK
mmcv: OK
timm: OK


## 1. 读取训练配置

训练脚本从 `config/train_dome.py` 读取配置。这里重点看和一个样本流动有关的字段：

- `return_len_train`：训练时输入帧数。
- `mid_frame/end_frame`：评估时历史帧和预测帧范围。
- `base_channel`：latent 通道数。
- `sample.sample_method='flow'`：使用 Flow Matching。
- `model.world_model.learn_sigma=False`：模型只输出速度场，不输出 diffusion 方差。

In [3]:
# 用 mmengine 读取配置；如果 mmengine 不可用，则退回 runpy。
try:
    from mmengine import Config
    cfg = Config.fromfile(str(PROJECT_ROOT / 'config' / 'train_dome.py'))
    get_cfg = lambda key, default=None: cfg.get(key, default)
except Exception as exc:
    print('mmengine 读取失败，使用 runpy 读取:', repr(exc))
    import runpy
    cfg = runpy.run_path(str(PROJECT_ROOT / 'config' / 'train_dome.py'))
    get_cfg = lambda key, default=None: cfg.get(key, default)

print('sample_method:', cfg['sample']['sample_method'])
print('num_sampling_steps:', cfg['sample']['num_sampling_steps'])
print('flow_sigma:', cfg['sample']['flow_sigma'])
print('model_time_scale:', cfg['sample']['model_time_scale'])
print('learn_sigma:', cfg['model']['world_model']['learn_sigma'])
print('return_len_train:', cfg['return_len_train'])
print('base_channel:', cfg['base_channel'])
print('p_use_pose_condition:', cfg['p_use_pose_condition'])

sample_method: flow
num_sampling_steps: 20
flow_sigma: 0.0
model_time_scale: 1000.0
learn_sigma: False
return_len_train: 11
base_channel: 64
p_use_pose_condition: 0.9


## 2. 一个训练 batch 的原始形状

训练循环中 dataloader 返回：

```python
for i_iter, (input_occs, target_occs, metas) in enumerate(train_dataset_loader):
```

`input_occs` 的典型形状是：

```text
(B, F, H, W, D)
```

在 DOME 配置里通常是：

```text
B = train_loader.batch_size
F = return_len_train = 11
H = 200
W = 200
D = 16
```

下面先用一个小尺寸 dummy occupancy 来模拟这个 batch，避免 notebook 占用太多内存。

In [4]:
import torch

# 为了演示，用小尺寸代替真实 200x200x16。
B = 2
F = int(cfg['return_len_train'])
H, W, D = 20, 20, 4
num_classes = 18

# occupancy 是离散语义标签，真实数据通常是 long/int 类型。
input_occs = torch.randint(0, num_classes, (B, F, H, W, D))
target_occs = input_occs.clone()
metas = [{'rel_poses': torch.zeros(F, 2)} for _ in range(B)]

print('input_occs:', tuple(input_occs.shape), input_occs.dtype)
print('target_occs:', tuple(target_occs.shape), target_occs.dtype)
print('metas[0] keys:', list(metas[0].keys()))

input_occs: (2, 11, 20, 20, 4) torch.int64
target_occs: (2, 11, 20, 20, 4) torch.int64
metas[0] keys: ['rel_poses']


## 3. VAE 编码：occupancy -> latent

真实训练代码：

```python
x = input_occs.cuda()
bs = x.shape[0]
x, shape = vae.forward_encoder(x)
x, _, _ = vae.sample_z(x)
x *= cfg.model.vae.scaling_factor
```

真实 VAE 会把每帧 occupancy 压到 latent，通道数通常是 `base_channel=64`，空间尺寸由 `200x200` 压到 `25x25`。

为了演示训练 pipeline，这里用一个 dummy VAE encoder 直接生成同样风格的 latent。

In [5]:
from einops import rearrange


class DummyVAE:
    """只模拟 VAE 输出 shape，不模拟真实重建质量。"""

    def __init__(self, latent_channels=64, latent_hw=5, scaling_factor=0.18215):
        self.latent_channels = latent_channels
        self.latent_hw = latent_hw
        self.scaling_factor = scaling_factor

    def forward_encoder(self, occs):
        b, f = occs.shape[:2]
        # 真实 Encoder2D 常见输出是 ((B*F), C, h, w)。
        z = torch.randn(b * f, self.latent_channels, self.latent_hw, self.latent_hw)
        shapes = [[20, 20], [10, 10], [5, 5]]
        return z, shapes

    def sample_z(self, z):
        # 真实 VAE 会从分布或量化结果中取样；这里直接返回。
        return z, None, None


vae = DummyVAE(latent_channels=int(cfg['base_channel']), latent_hw=5)

x, vae_shapes = vae.forward_encoder(input_occs)
x, _, _ = vae.sample_z(x)
x = x * vae.scaling_factor

print('VAE 原始 latent:', tuple(x.shape))
print('VAE shapes:', vae_shapes)

VAE 原始 latent: (22, 64, 5, 5)
VAE shapes: [[20, 20], [10, 10], [5, 5]]


## 4. 整理为 DOME 输入维度

DOME 的 `forward()` 需要：

```text
x: (B, F, C, H_latent, W_latent)
t: (B,)
```

所以训练脚本会把 VAE 输出从 `(B*F, C, h, w)` 还原为 `(B, F, C, h, w)`。

In [6]:
if x.dim() == 4:
    x = rearrange(x, '(b f) c h w -> b f c h w', b=B).contiguous()
elif x.dim() == 5:
    x = rearrange(x, 'b c f h w -> b f c h w', b=B).contiguous()
else:
    raise NotImplementedError(x.shape)

print('DOME 输入 latent x:', tuple(x.shape))
print('含义: (B, F, C, H_latent, W_latent)')

DOME 输入 latent x: (2, 11, 64, 5, 5)
含义: (B, F, C, H_latent, W_latent)


## 5. 准备条件信息 `model_kwargs`

训练脚本中 pose 条件不是每次都用：

```python
use_pose_condition = torch.rand(1) < cfg.p_use_pose_condition
model_kwargs = {}
if use_pose_condition:
    model_kwargs['metas'] = metas
```

这相当于一种条件 dropout。让模型既能学会带 pose 条件的生成，也能在缺少条件时工作。

In [7]:
p_use_pose_condition = float(cfg['p_use_pose_condition'])
use_pose_condition = torch.rand(1).item() < p_use_pose_condition

model_kwargs = {}
if use_pose_condition:
    model_kwargs['metas'] = metas

print('use_pose_condition:', use_pose_condition)
print('model_kwargs keys:', list(model_kwargs.keys()))

use_pose_condition: True
model_kwargs keys: ['metas']


## 6. Flow Matching 训练目标

本项目新增的 `FlowMatching.training_losses()` 会做这些事：

1. 采样噪声 `x0 ~ N(0, I)`。
2. 令真实 latent 为 `x1`。
3. 使用 `torchcfm` 采样时间 `t ∈ [0, 1]`、中间点 `x_t`、目标速度 `u_t`。
4. 如果启用历史条件帧替换，将部分帧直接替换为真实 latent。
5. 调用 DOME：`model(x_t, model_time_scale * t, **model_kwargs)`。
6. 用 `MSE(model_output, u_t)` 作为 loss。

注意：DOME 原来的 timestep embedding 习惯接收类似 diffusion 的 0~1000 时间尺度，所以这里默认 `model_time_scale=1000.0`。

In [8]:
from diffusion import create_flow_matching

flow = create_flow_matching(
    num_sampling_steps=int(cfg['sample']['num_sampling_steps']),
    sigma=float(cfg['sample']['flow_sigma']),
    replace_cond_frames=bool(cfg['replace_cond_frames']),
    cond_frames_choices=cfg['cond_frames_choices'],
    model_time_scale=float(cfg['sample']['model_time_scale']),
)

print('flow.num_timesteps:', flow.num_timesteps)
print('flow.model_time_scale:', flow.model_time_scale)
print('replace_cond_frames:', flow.replace_cond_frames)
print('cond_frames_choices:', flow.cond_frames_choices)

flow.num_timesteps: 20
flow.model_time_scale: 1000.0
replace_cond_frames: True
cond_frames_choices: [[], [0], [0, 1], [0, 1, 2], [0, 1, 2, 3]]


## 7. 用 dummy DOME 计算一次 loss

真实 DOME 输出速度场，shape 必须等于 latent shape：

```text
model_output.shape == x.shape == (B, F, C, h, w)
```

这里用一个简单线性比例模型模拟输出，目的是验证训练接口和 loss shape。

In [9]:
class DummyDOME(torch.nn.Module):
    """最小速度场模型：只用于检查训练 pipeline。"""

    def __init__(self):
        super().__init__()
        self.scale = torch.nn.Parameter(torch.tensor(0.0))

    def forward(self, x_t, t, **kwargs):
        # x_t: (B, F, C, h, w)
        # t: (B,)，已被 FlowMatching 放大到 0~1000。
        assert x_t.ndim == 5
        assert t.shape == (x_t.shape[0],)
        return self.scale * x_t


my_model = DummyDOME()

# 训练脚本里还会采样一个 t，但 flow matching 内部会自己采样连续 t；这个参数保留只是为了接口兼容。
t_placeholder = torch.randint(0, flow.num_timesteps, (x.shape[0],))

loss_dict = flow.training_losses(my_model, x, t_placeholder, model_kwargs=model_kwargs)
loss = loss_dict['loss'].mean()

print('loss_dict keys:', sorted(loss_dict.keys()))
print('loss batch shape:', tuple(loss_dict['loss'].shape))
print('loss mean:', float(loss.detach()))

loss_dict keys: ['loss', 'mse']
loss batch shape: (2,)
loss mean: 0.8031951189041138


## 8. 反向传播与优化器更新

真实训练代码使用 AMP、梯度裁剪和 scheduler：

```python
optimizer.zero_grad()
scaler.scale(loss).backward()
scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(my_model.parameters(), cfg.grad_max_norm)
scaler.step(optimizer)
scaler.update()
scheduler.step_update(global_iter)
```

下面用普通 CPU backward 演示参数确实能收到梯度。

In [10]:
optimizer = torch.optim.AdamW(my_model.parameters(), lr=1e-4, weight_decay=1e-4)

optimizer.zero_grad()
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(my_model.parameters(), max_norm=float(cfg['grad_max_norm']))
optimizer.step()

print('grad_norm:', float(grad_norm))
print('updated scale:', float(my_model.scale.detach()))

grad_norm: 0.4590054154396057
updated scale: -9.999999747378752e-05


## 9. 评估/采样阶段的单样本流程

训练后的评估阶段会：

1. 用 VAE 编码输入帧，得到 `input_latents`。
2. 从高斯噪声初始化完整预测序列 latent。
3. 使用 `flow.p_sample_loop()` 做 ODE 积分。
4. 每一步都把历史条件帧替换回真实 latent。
5. 用 VAE decoder 把 latent 解码回 occupancy logits。

下面只演示 latent 采样和条件帧保持。

In [11]:
n_conds = int(cfg['sample']['n_conds'])
initial_cond_indices = list(range(n_conds)) if n_conds else None

# 这里 shape 与 x 相同，真实评估里通常是 (B, end_frame, C, h, w)。
sample = flow.p_sample_loop(
    my_model,
    shape=x.shape,
    noise=torch.randn_like(x),
    device='cpu',
    model_kwargs=model_kwargs,
    initial_cond_indices=initial_cond_indices,
    initial_cond_frames=x,
)

print('sample:', tuple(sample.shape))
if initial_cond_indices:
    kept = torch.allclose(sample[:, initial_cond_indices], x[:, initial_cond_indices])
    print('历史条件帧是否保持:', kept)
else:
    print('没有设置条件帧。')

sample: (2, 11, 64, 5, 5)
历史条件帧是否保持: True


## 10. 真实单 batch 调试模板

下面是接入真实 dataloader / VAE / DOME 的模板。默认注释掉，等你准备好数据和 checkpoint 后再逐段取消注释。

这段代码的目标不是完整训练，而是只跑一个 batch，确认真实 pipeline 可以走到 flow matching loss。

In [ ]:
# 真实单 batch 调试模板：需要数据、ckpt 和完整依赖。
#
# import torch
# from mmengine import Config
# from mmengine.registry import MODELS
# from einops import rearrange
# import model
# from dataset import get_dataloader
# from diffusion import create_flow_matching
#
# cfg = Config.fromfile(str(PROJECT_ROOT / 'config' / 'train_dome.py'))
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
#
# my_model = MODELS.build(cfg.model.world_model).to(device).train()
# vae = MODELS.build(cfg.model.vae).to(device).eval()
# vae.requires_grad_(False)
#
# flow = create_flow_matching(
#     num_sampling_steps=cfg.sample.num_sampling_steps,
#     sigma=cfg.sample.flow_sigma,
#     replace_cond_frames=cfg.replace_cond_frames,
#     cond_frames_choices=cfg.cond_frames_choices,
#     model_time_scale=cfg.sample.model_time_scale,
# )
#
# train_loader, val_loader = get_dataloader(
#     cfg.train_dataset_config,
#     cfg.val_dataset_config,
#     cfg.train_wrapper_config,
#     cfg.val_wrapper_config,
#     cfg.train_loader,
#     cfg.val_loader,
#     dist=False,
# )
#
# input_occs, target_occs, metas = next(iter(train_loader))
# input_occs = input_occs.to(device)
# bs = input_occs.shape[0]
#
# with torch.no_grad():
#     x, shapes = vae.forward_encoder(input_occs)
#     x, _, _ = vae.sample_z(x)
#     x = x * cfg.model.vae.scaling_factor
#     if x.dim() == 4:
#         x = rearrange(x, '(b f) c h w -> b f c h w', b=bs).contiguous()
#     elif x.dim() == 5:
#         x = rearrange(x, 'b c f h w -> b f c h w', b=bs).contiguous()
#     else:
#         raise NotImplementedError(x.shape)
#
# model_kwargs = {'metas': metas}
# loss_dict = flow.training_losses(my_model, x, model_kwargs=model_kwargs)
# print({k: float(v.mean()) for k, v in loss_dict.items()})

## 11. 一句话总结

对一个训练样本来说，本次改动只改变了生成模型训练目标：

```text
旧 diffusion:  给 x 加噪，模型预测噪声 epsilon 或 x0
新 flow:       在噪声 x0 和数据 x1 之间采样 x_t，模型预测速度 u_t
```

VAE 编码、DOME 的 `(x, t, metas)` 输入形式、optimizer step、EMA、评估时的条件帧替换逻辑都沿用原来的 pipeline。